In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim

In [2]:
import pandas as pd

df = pd.read_csv('nyc_traffic_congestion.csv')
df

,Boro,Vol,geometry,congestion,congestion_level,timestamp
0,Brooklyn,252,POLYGON ((-73.97028542436098 40.67474068522885...,0.680168,0,6/16/2025 19:45
1,Brooklyn,204,POLYGON ((-73.97028542436098 40.67474068522885...,0.136034,0,6/16/2025 20:00
2,Brooklyn,257,POLYGON ((-73.97028542436098 40.67474068522885...,0.736849,0,6/16/2025 20:15
3,Brooklyn,179,POLYGON ((-73.97028542436098 40.67474068522885...,-0.147370,0,6/16/2025 20:30
4,Brooklyn,192,POLYGON ((-73.97028542436098 40.67474068522885...,0.000000,0,6/16/2025 20:45
...,...,...,...,...,...,...
148797,Brooklyn,120,POLYGON ((-73.93800729996994 40.65260186906368...,1.349000,1,5/10/2024 16:30
148798,Brooklyn,95,POLYGON ((-73.93800729996994 40.65260186906368...,0.615848,0,5/10/2024 16:45
148799,Brooklyn,119,POLYGON ((-73.93800729996994 40.65260186906368...,1.319674,1,5/10/2024 17:00
148800,Brooklyn,114,POLYGON ((-73.93800729996994 40.65260186906368...,1.173043,1,5/10/2024 17:15


In [3]:
df = df[['congestion_level', 'timestamp']]
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
for dataset in [train_df, test_df]:
    dataset['month'] = dataset['timestamp'].dt.month
    dataset['day'] = dataset['timestamp'].dt.day
    dataset['day_of_week'] = dataset['timestamp'].dt.dayofweek
    dataset['hour'] = dataset['timestamp'].dt.hour
    dataset['minute'] = dataset['timestamp'].dt.minute
features = ['month', 'day', 'day_of_week', 'hour', 'minute']
X_train = torch.FloatTensor(train_df[features].values)
y_train = torch.FloatTensor(train_df['congestion_level'].values).unsqueeze(1)
X_test = torch.FloatTensor(test_df[features].values)
y_test = torch.FloatTensor(test_df['congestion_level'].values).unsqueeze(1)

/tmp/ipykernel_14960/2038528523.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df['timestamp'])


In [4]:
num_classes=3
class lstmTrafficClassifier(nn.Module):
    def __init__(self, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = None
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)

        batch_size, seq_len, input_size = x.shape
        if self.lstm is None:
            self.lstm = nn.LSTM(
                input_size=input_size,
                hidden_size=self.hidden_size,
                num_layers=self.num_layers,
                batch_first=True,
                dropout=0.2 if self.num_layers > 1 else 0.0
            ).to(x.device)

        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out
model = lstmTrafficClassifier(hidden_size=64, num_layers=2, num_classes=num_classes)
y_train_int = y_train.view(-1).long()
class_counts = np.bincount(y_train_int.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights[2] = class_weights[2] * 1.3
class_weights = torch.FloatTensor(class_weights)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [5]:
epochs = 50
batch_size = 32

for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train[indices], y_train[indices]
        batch_y = batch_y.view(-1).long()

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/50, Loss: 0.8605
Epoch 2/50, Loss: 1.2165
Epoch 3/50, Loss: 1.1205
Epoch 4/50, Loss: 0.9828
Epoch 5/50, Loss: 0.8268
Epoch 6/50, Loss: 0.8635
Epoch 7/50, Loss: 1.0161
Epoch 8/50, Loss: 0.7794
Epoch 9/50, Loss: 0.7317
Epoch 10/50, Loss: 0.6886
Epoch 11/50, Loss: 0.6680
Epoch 12/50, Loss: 1.3832
Epoch 13/50, Loss: 0.9298
Epoch 14/50, Loss: 0.9496
Epoch 15/50, Loss: 0.9018
Epoch 16/50, Loss: 1.4873
Epoch 17/50, Loss: 1.5302
Epoch 18/50, Loss: 0.9086
Epoch 19/50, Loss: 0.6767
Epoch 20/50, Loss: 0.9666
Epoch 21/50, Loss: 1.0967
Epoch 22/50, Loss: 1.2444
Epoch 23/50, Loss: 0.6835
Epoch 24/50, Loss: 1.3861
Epoch 25/50, Loss: 0.7818
Epoch 26/50, Loss: 0.9994
Epoch 27/50, Loss: 0.8033
Epoch 28/50, Loss: 0.8946
Epoch 29/50, Loss: 0.8749
Epoch 30/50, Loss: 1.2940
Epoch 31/50, Loss: 1.3364
Epoch 32/50, Loss: 1.3995
Epoch 33/50, Loss: 1.0937
Epoch 34/50, Loss: 0.6902
Epoch 35/50, Loss: 1.5893
Epoch 36/50, Loss: 1.1017
Epoch 37/50, Loss: 1.8590
Epoch 38/50, Loss: 0.8150
Epoch 39/50, Loss: 1.

In [6]:
from sklearn.metrics import f1_score, accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
model.eval()
with torch.no_grad():
    logits = model(X_test)
    y_pred = torch.argmax(logits, dim=1)
f1 = f1_score(y_test.numpy(), y_pred.numpy(), average='micro')
precision = precision_score(y_test.numpy(), y_pred.numpy(), average='macro')
recall = recall_score(y_test.numpy(), y_pred.numpy(), average='macro')
cm = classification_report(y_test.numpy(), y_pred.numpy())

print(f1)
print(precision)
print(recall)
print(cm)

0.5319549293250599
0.3601985116542404
0.37461467057961206
              precision    recall  f1-score   support

         0.0       0.91      0.57      0.70     38809
         1.0       0.14      0.32      0.20      4491
         2.0       0.03      0.24      0.05      1341

    accuracy                           0.53     44641
   macro avg       0.36      0.37      0.32     44641
weighted avg       0.80      0.53      0.63     44641



In [ ]:
from sklearn.model_selection import KFold
import itertools
import random
param_grid = {
    'hidden_size': [32, 64, 128, 256],
    'num_layers': [1, 2, 3],
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [32, 64, 128]
}

all_combinations = list(itertools.product(
    param_grid['hidden_size'],
    param_grid['num_layers'],
    param_grid['learning_rate'],
    param_grid['batch_size']
))

random.seed(42)
selected_combinations = random.sample(all_combinations, 40)
print("Number of hyperparameter combinations selected:", len(selected_combinations))

num_folds = 3
best_f1 = 0
best_params = None
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

for hidden_size, num_layers, lr, batch_size in selected_combinations:
    fold_f1_scores = []
    print(f"\nTesting hyperparams: hidden_size={hidden_size}, num_layers={num_layers}, lr={lr}, batch_size={batch_size}")

    for train_index, val_index in kf.split(X_train):
        X_train_fold = X_train[train_index]
        y_train_fold = y_train[train_index].view(-1).long()
        X_val_fold = X_train[val_index]
        y_val_fold = y_train[val_index].view(-1).long()
        model = lstmTrafficClassifier(hidden_size=hidden_size, num_layers=num_layers, num_classes=num_classes)
        class_counts = np.bincount(y_train_fold.numpy())
        class_weights = 1.0 / class_counts
        class_weights = class_weights / class_weights.sum() * len(class_counts)
        class_weights[1] = class_weights[1] * 1.3
        class_weights[2] = class_weights[2] * 1.3
        class_weights = torch.FloatTensor(class_weights)

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        epochs = 20
        for epoch in range(epochs):
            model.train()
            permutation = torch.randperm(X_train_fold.size(0))
            for i in range(0, X_train_fold.size(0), batch_size):
                batch_indices = permutation[i:i+batch_size]
                batch_x = X_train_fold[batch_indices]
                batch_y = y_train_fold[batch_indices]

                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_fold)
            y_pred_val = torch.argmax(val_logits, dim=1)
            fold_f1 = f1_score(y_val_fold.numpy(), y_pred_val.numpy(), average='micro')
            fold_f1_scores.append(fold_f1)

    mean_f1 = np.mean(fold_f1_scores)
    print(f"Mean 3-fold CV precision: {mean_f1:.4f}")
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_params = {'hidden_size': hidden_size, 'num_layers': num_layers, 'learning_rate': lr, 'batch_size': batch_size}
        torch.save(model.state_dict(), "best_lstm_classifier.pth")

print("\nBest Hyperparameters:", best_params)
print("Best 3-fold CV precision:", best_f1)

Number of hyperparameter combinations selected: 40

Testing hyperparams: hidden_size=256, num_layers=1, lr=0.001, batch_size=32
Epoch 1/20, Loss: 0.9653
Epoch 2/20, Loss: 0.8389
Epoch 3/20, Loss: 0.8614
Epoch 4/20, Loss: 1.1603
Epoch 5/20, Loss: 0.9740
Epoch 6/20, Loss: 1.0263
Epoch 7/20, Loss: 1.0218
Epoch 8/20, Loss: 0.9468
Epoch 9/20, Loss: 0.8923
Epoch 10/20, Loss: 0.8858
Epoch 11/20, Loss: 0.7606
Epoch 12/20, Loss: 0.7892
Epoch 13/20, Loss: 0.7732
Epoch 14/20, Loss: 1.1277
Epoch 15/20, Loss: 0.8797
Epoch 16/20, Loss: 1.1413
Epoch 17/20, Loss: 1.0049
Epoch 18/20, Loss: 0.7166
Epoch 19/20, Loss: 0.8471
Epoch 20/20, Loss: 0.8857
Epoch 1/20, Loss: 1.2438
Epoch 2/20, Loss: 1.2060
Epoch 3/20, Loss: 1.1486
Epoch 4/20, Loss: 1.0415
Epoch 5/20, Loss: 0.8916
Epoch 6/20, Loss: 1.1598
Epoch 7/20, Loss: 1.0309
Epoch 8/20, Loss: 0.8155
Epoch 9/20, Loss: 0.6637
Epoch 10/20, Loss: 1.6517
Epoch 11/20, Loss: 0.2413
Epoch 12/20, Loss: 1.2756
Epoch 13/20, Loss: 0.0431
Epoch 14/20, Loss: 1.0308
Epoch 

In [ ]:
best_hidden_size = best_params['hidden_size']
best_num_layers = best_params['num_layers']
best_lr = best_params['learning_rate']
best_batch_size = best_params['batch_size']

final_model = lstmTrafficClassifier(hidden_size=best_hidden_size,
                                   num_layers=best_num_layers,
                                   num_classes=num_classes)

y_train_flat = y_train.view(-1).long()
class_counts = np.bincount(y_train_flat.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights[2] = class_weights[2] * 1.3
class_weights = torch.FloatTensor(class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(final_model.parameters(), lr=best_lr)

epochs = 30
for epoch in range(epochs):
    final_model.train()
    permutation = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), best_batch_size):
        batch_indices = permutation[i:i+best_batch_size]
        batch_x = X_train[batch_indices]
        batch_y = y_train_flat[batch_indices]

        optimizer.zero_grad()
        outputs = final_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

final_model.eval()
with torch.no_grad():
    test_logits = final_model(X_test)
    y_pred_test = torch.argmax(test_logits, dim=1)

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

f1 = f1_score(y_test.view(-1).numpy(), y_pred_test.numpy(), average='micro')
precision = precision_score(y_test.view(-1).numpy(), y_pred_test.numpy(), average='macro')
recall = recall_score(y_test.view(-1).numpy(), y_pred_test.numpy(), average='macro')
cm = classification_report(y_test.view(-1).numpy(), y_pred_test.numpy())

print(f1)
print(precision)
print(recall)
print(cm)

Epoch 1/30, Loss: 0.9517
Epoch 2/30, Loss: 1.4832
Epoch 3/30, Loss: 1.5389
Epoch 4/30, Loss: 0.6955
Epoch 5/30, Loss: 0.2579
Epoch 6/30, Loss: 0.9572
Epoch 7/30, Loss: 0.0865
Epoch 8/30, Loss: 1.3228
Epoch 9/30, Loss: 1.2686
Epoch 10/30, Loss: 0.9683
Epoch 11/30, Loss: 1.1748
Epoch 12/30, Loss: 1.1743
Epoch 13/30, Loss: 1.3117
Epoch 14/30, Loss: 0.0436
Epoch 15/30, Loss: 0.2318
Epoch 16/30, Loss: 1.2253
Epoch 17/30, Loss: 0.7987
Epoch 18/30, Loss: 0.7007
Epoch 19/30, Loss: 1.5055
Epoch 20/30, Loss: 1.0855
Epoch 21/30, Loss: 0.9589
Epoch 22/30, Loss: 1.2852
Epoch 23/30, Loss: 1.4560
Epoch 24/30, Loss: 1.5595
Epoch 25/30, Loss: 1.0910
Epoch 26/30, Loss: 1.8413
Epoch 27/30, Loss: 0.7932
Epoch 28/30, Loss: 0.0541
Epoch 29/30, Loss: 0.4006
Epoch 30/30, Loss: 0.0902
0.5331645796465133
0.3910926388071055
0.4480645961343968
              precision    recall  f1-score   support

         0.0       0.96      0.54      0.69     38809
         1.0       0.18      0.59      0.28      4491
         

In [ ]:
class TrainedlstmTimeEncoder(nn.Module):
    def __init__(self, trained_lstm_model, embedding_size=32):
        super().__init__()
        self.lstm = trained_lstm_model.lstm
        self.hidden_size = trained_lstm_model.hidden_size
        self.embedding_size = embedding_size
        self.fc_embed = nn.Linear(self.hidden_size, embedding_size)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)

        batch_size, seq_len, input_size = x.shape
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        embedding = self.fc_embed(last_hidden)
        return embedding

time_encoder = TrainedlstmTimeEncoder(model, embedding_size=32)

time_embeddings = time_encoder(X_train[:5])
print("Time embedding shape:", time_embeddings.shape)


In [ ]:
time_encoder.eval()
sample_batch = X_train[:5]
with torch.no_grad():
    embeddings = time_encoder(sample_batch)
print("Embeddings shape:", embeddings.shape)
print("Embeddings:")
print(embeddings)

In [ ]:
model.eval()
input_size = X_train.shape[-1]
example_input = torch.randn(1, input_size)
scripted_model = torch.jit.trace(time_encoder, example_input)

torch.jit.save(scripted_model, "time_weights.pt")

In [ ]:
torch.save(model, "time_weights.pth")


In [ ]:
from google.colab import files

files.download('time_weights.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>